# BLIP-2 Flickr8k Smoke Train on Colab

This notebook is designed for **Colab free tier**.

It does a short smoke test to answer three questions before larger training:

1. Does the model load correctly?
2. Does a short training run complete without breaking?
3. Does caption generation work after training?

Model path used here:
- PVT v2 b2 vision encoder
- Q-Former LoRA
- flan-t5-small for Colab-friendly memory usage

This is a **sanity-check notebook**, not final training.

## Run Order

Run the notebook from top to bottom.

If you are a beginner, do not change anything on the first run.
Only change settings after you get one full successful run.

## What You Need

1. A Colab GPU runtime.
2. A Kaggle account.
3. Flickr8k download access from Kaggle.
4. Either:
   - `KAGGLE_USERNAME` and `KAGGLE_KEY` in Colab Secrets, or
   - your Kaggle username and key ready to paste, or
   - `kaggle.json` ready to upload.

## Beginner Setup Notes

If Colab asks to restart the runtime after installing packages, restart it and then continue from the next cell.

If the Kaggle step fails, the easiest fix is:
1. Open Kaggle.
2. Go to Settings.
3. Create a new API token.
4. Either paste the username/key when prompted or upload the downloaded `kaggle.json` file.

On free-tier Colab, a T4 GPU is usually enough for this smoke test.

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers==4.46.2 peft==0.13.2 timm==1.0.15 omegaconf==2.3.0 iopath fairscale sentencepiece einops kaggle pandas pillow

In [ ]:
# Cell 1b: Load API token from Colab Secrets
import os

try:
    from google.colab import userdata
    api_key = userdata.get('API_TOKEN')
    os.environ.setdefault('HF_TOKEN', api_key)
    os.environ.setdefault('HUGGING_FACE_HUB_TOKEN', api_key)
    print('API_TOKEN loaded from Colab Secrets.')
except Exception:
    api_key = None
    print('API_TOKEN not found in Colab Secrets (optional).')

In [ ]:
# Cell 2: Clone the repo branch used for this work
import os
import sys
import shutil
from pathlib import Path

REPO_URL = 'https://github.com/archi-dev1/lavis-ai.git'
REPO_BRANCH = 'pre-training-stable'
REPO_DIR = Path('/content/lavis-ai')

if not REPO_DIR.exists():
    !git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} /content/lavis-ai
else:
    print(f'Reusing existing repo at {REPO_DIR}')

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Repo ready:', REPO_DIR)

In [ ]:
# Cell 3: Patch Blip2T5.generate() so no_repeat_ngram_size works
from pathlib import Path

target = Path('/content/lavis-ai/lavis/models/blip2_models/blip2_t5.py')
text = target.read_text()

if 'no_repeat_ngram_size=0' not in text:
    old_signature = '''    def generate(
        self,
        samples,
        use_nucleus_sampling=False,
        num_beams=5,
        max_length=30,
        min_length=1,
        top_p=0.9,
        repetition_penalty=1.0,
        length_penalty=1.0,
        num_captions=1,
        temperature=1,
    ):'''
    new_signature = '''    def generate(
        self,
        samples,
        use_nucleus_sampling=False,
        num_beams=5,
        max_length=30,
        min_length=1,
        top_p=0.9,
        repetition_penalty=1.0,
        length_penalty=1.0,
        num_captions=1,
        temperature=1,
        no_repeat_ngram_size=0,
    ):'''
    text = text.replace(old_signature, new_signature)

    old_call = '''                repetition_penalty=repetition_penalty,
                length_penalty=length_penalty,
                num_return_sequences=num_captions,
            )'''
    new_call = '''                repetition_penalty=repetition_penalty,
                length_penalty=length_penalty,
                num_return_sequences=num_captions,
                no_repeat_ngram_size=no_repeat_ngram_size,
            )'''
    text = text.replace(old_call, new_call)
    target.write_text(text)
    print('Patched Blip2T5.generate().')
else:
    print('Patch already present.')

## Kaggle Auth

The notebook will try Kaggle credentials in this order:

1. Existing `~/.kaggle/kaggle.json`
2. `KAGGLE_USERNAME` and `KAGGLE_KEY` from environment variables
3. `KAGGLE_USERNAME` and `KAGGLE_KEY` from Colab Secrets
4. A direct username/key prompt inside the notebook
5. `kaggle.json` upload as the last fallback

For beginners, the easiest options are usually:
- use Colab Secrets, or
- paste the username and key when the notebook asks for them.

In [ ]:
# Cell 4: Set up Kaggle credentials and download Flickr8k
from pathlib import Path
import os
from getpass import getpass

DATA_ROOT = Path('/content/data/flickr8k')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json = kaggle_dir / 'kaggle.json'

if kaggle_json.exists():
    print(f'Using existing {kaggle_json}')
else:
    username = os.environ.get('KAGGLE_USERNAME')
    key = os.environ.get('KAGGLE_KEY')

    if not username or not key:
        try:
            from google.colab import userdata
            username = username or userdata.get('KAGGLE_USERNAME')
            key = key or userdata.get('KAGGLE_KEY')
        except Exception:
            pass

    if username and key:
        kaggle_json.write_text(
            '{\n'
            f'  "username": "{username}",\n'
            f'  "key": "{key}"\n'
            '}\n'
        )
        print('Created ~/.kaggle/kaggle.json from env vars or Colab Secrets')
    else:
        print('No Kaggle credentials found in ~/.kaggle, env vars, or Colab Secrets.')
        print('Paste them below, or press Enter on username to upload kaggle.json instead.')

        entered_username = input('Kaggle username: ').strip()
        entered_key = getpass('Kaggle key: ').strip() if entered_username else ''

        if entered_username and entered_key:
            kaggle_json.write_text(
                '{\n'
                f'  "username": "{entered_username}",\n'
                f'  "key": "{entered_key}"\n'
                '}\n'
            )
            print('Created ~/.kaggle/kaggle.json from pasted credentials')
        else:
            from google.colab import files
            print('Upload kaggle.json downloaded from Kaggle > Settings > Create New Token.')
            uploaded = files.upload()
            if 'kaggle.json' not in uploaded:
                raise RuntimeError('kaggle.json not uploaded')
            kaggle_json.write_bytes(uploaded['kaggle.json'])

kaggle_json.chmod(0o600)

has_images = any(DATA_ROOT.rglob('*.jpg')) or any(DATA_ROOT.rglob('*.png'))
if not has_images:
    !kaggle datasets download -d adityajn105/flickr8k -p /content/data/flickr8k --unzip
else:
    print('Reusing existing Flickr8k download')

In [ ]:
# Cell 5: Build dataset helpers and smoke-test settings
import csv
import random
import types
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import transforms

SEED = 42
TRAIN_IMAGE_LIMIT = 7000
VAL_IMAGE_LIMIT = 500
TEST_IMAGE_LIMIT = 500
CAPTIONS_PER_IMAGE = 1
IMAGE_SIZE = 224
BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
NUM_WORKERS = 2
MAX_STEPS = 100000
EPOCHS = 20
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_BEAMS = 3
MAX_LENGTH = 20
NO_REPEAT_NGRAM_SIZE = 3
USE_NUCLEUS_SAMPLING = True
TOP_P = 0.9
TEMPERATURE = 1.0
REPETITION_PENALTY = 1.5
OUTPUT_DIR = Path('/content/outputs/flickr8k_smoke')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if hasattr(torch, 'set_float32_matmul_precision'):
    torch.set_float32_matmul_precision('high')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2))

def find_first(root, names):
    root = Path(root)
    names = {name.lower() for name in names}
    for path in root.rglob('*'):
        if path.is_file() and path.name.lower() in names:
            return path
    return None

def find_images_dir(root):
    candidates = []
    root = Path(root)
    for path in root.rglob('*'):
        if path.is_dir():
            count = len(list(path.glob('*.jpg'))) + len(list(path.glob('*.jpeg'))) + len(list(path.glob('*.png')))
            if count >= 100:
                candidates.append((count, path))
    if not candidates:
        raise RuntimeError('Could not find image directory')
    candidates.sort(reverse=True)
    return candidates[0][1]

def read_split_file(base_dir, filename):
    path = find_first(base_dir, {filename})
    if path is None:
        return None
    return [line.strip() for line in path.read_text().splitlines() if line.strip()]

def parse_flickr8k_captions(captions_file, images_dir):
    captions_by_image = defaultdict(list)
    suffix = captions_file.suffix.lower()

    if captions_file.name.lower() == 'flickr8k.token.txt':
        for line in captions_file.read_text(encoding='utf-8').splitlines():
            if not line.strip():
                continue
            image_part, caption = line.split('\t', 1)
            image_name = image_part.split('#', 1)[0].strip()
            captions_by_image[image_name].append(caption.strip())
    elif suffix == '.csv':
        import pandas as pd
        frame = pd.read_csv(captions_file)
        columns = {col.lower(): col for col in frame.columns}
        image_col = columns.get('image') or columns.get('filename') or frame.columns[0]
        caption_col = columns.get('caption') or columns.get('raw') or frame.columns[1]
        for _, row in frame.iterrows():
            captions_by_image[str(row[image_col]).strip()].append(str(row[caption_col]).strip())
    else:
        with captions_file.open('r', encoding='utf-8') as handle:
            sample = handle.read(2048)
            handle.seek(0)
            dialect = csv.Sniffer().sniff(sample)
            reader = csv.reader(handle, dialect)
            rows = list(reader)
        if rows and rows[0] and rows[0][0].lower() in {'image', 'filename'}:
            rows = rows[1:]
        for row in rows:
            if len(row) < 2:
                continue
            captions_by_image[row[0].strip()].append(row[1].strip())

    available = {}
    for image_name, captions in captions_by_image.items():
        if (images_dir / image_name).exists() and captions:
            available[image_name] = captions

    if not available:
        raise RuntimeError('No valid image/caption pairs found')

    return available

def build_splits(base_dir, available_captions):
    train_names = read_split_file(base_dir, 'Flickr_8k.trainImages.txt')
    val_names = read_split_file(base_dir, 'Flickr_8k.devImages.txt')
    test_names = read_split_file(base_dir, 'Flickr_8k.testImages.txt')

    if train_names and val_names and test_names:
        all_names = [name for name in train_names + val_names + test_names if name in available_captions]
        random.shuffle(all_names)
        train_names = all_names[:TRAIN_IMAGE_LIMIT]
        val_names = all_names[TRAIN_IMAGE_LIMIT:TRAIN_IMAGE_LIMIT + VAL_IMAGE_LIMIT]
        test_names = all_names[TRAIN_IMAGE_LIMIT + VAL_IMAGE_LIMIT:TRAIN_IMAGE_LIMIT + VAL_IMAGE_LIMIT + TEST_IMAGE_LIMIT]
    else:
        names = sorted(available_captions.keys())
        random.shuffle(names)
        train_names = names[:TRAIN_IMAGE_LIMIT]
        val_names = names[TRAIN_IMAGE_LIMIT:TRAIN_IMAGE_LIMIT + VAL_IMAGE_LIMIT]
        test_names = names[TRAIN_IMAGE_LIMIT + VAL_IMAGE_LIMIT:TRAIN_IMAGE_LIMIT + VAL_IMAGE_LIMIT + TEST_IMAGE_LIMIT]

    return train_names, val_names, test_names

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

images_dir = find_images_dir(DATA_ROOT)
captions_file = find_first(DATA_ROOT, {'captions.txt', 'captions.csv', 'flickr8k.token.txt'})
if captions_file is None:
    text_files = list(Path(DATA_ROOT).rglob('*.txt')) + list(Path(DATA_ROOT).rglob('*.csv'))
    if not text_files:
        raise RuntimeError('Could not find captions file')
    captions_file = text_files[0]

captions_by_image = parse_flickr8k_captions(captions_file, images_dir)
train_names, val_names, test_names = build_splits(DATA_ROOT, captions_by_image)

print('Images dir:', images_dir)
print('Captions file:', captions_file)
print('Train images:', len(train_names))
print('Val images:', len(val_names))

print('Test images:', len(test_names))

In [ ]:
# Cell 6: Build the dataset and dataloader
class Flickr8kCaptionDataset:
    def __init__(self, image_dir, captions_by_image, image_names, transform, prompt='a photo of '):
        self.image_dir = Path(image_dir)
        self.transform = transform
        self.prompt = prompt
        self.samples = []
        for image_name in image_names:
            for caption in captions_by_image[image_name][:CAPTIONS_PER_IMAGE]:
                self.samples.append((image_name, caption.strip()))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        image_name, caption = self.samples[index]
        image_path = self.image_dir / image_name
        with Image.open(image_path) as image:
            image = image.convert('RGB')
            image_tensor = self.transform(image)
        return {
            'image': image_tensor,
            'text_input': self.prompt,
            'text_output': caption,
            'image_name': image_name,
        }

def collate_fn(batch):
    return {
        'image': torch.stack([item['image'] for item in batch], dim=0),
        'text_input': [item['text_input'] for item in batch],
        'text_output': [item['text_output'] for item in batch],
        'image_name': [item['image_name'] for item in batch],
    }

train_dataset = Flickr8kCaptionDataset(images_dir, captions_by_image, train_names, transform)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn,
)

print('Train pairs:', len(train_dataset))
print('Steps per epoch:', len(train_loader))

In [ ]:
# Cell 7: Load BLIP-2 with PVT + Q-Former LoRA
for pkg in [
    'lavis',
    'lavis.datasets',
    'lavis.datasets.builders',
    'lavis.processors',
    'lavis.tasks',
    'lavis.runners',
    'lavis.models',
]:
    if pkg not in sys.modules:
        module = types.ModuleType(pkg)
        module.__path__ = [str(REPO_DIR / pkg.replace('.', '/'))]
        module.__package__ = pkg
        sys.modules[pkg] = module

import lavis.common.registry
import lavis.common.utils
from lavis.models.base_model import BaseModel
sys.modules['lavis.models'].BaseModel = BaseModel

from lavis.models.blip2_models.blip2 import apply_lora_to_qformer, freeze_qformer_base, get_trainable_params_info
from lavis.models.blip2_models.blip2_t5 import Blip2T5

model = Blip2T5(
    vit_model='pvt_v2_b2',
    img_size=IMAGE_SIZE,
    num_query_token=16,
    t5_model='google/flan-t5-small',
    drop_path_rate=0,
    use_grad_checkpoint=False,
    vit_precision='fp32',
    freeze_vit=True,
    prompt='a photo of ',
    max_txt_len=32,
)

model.Qformer = apply_lora_to_qformer(
    model.Qformer,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=['query', 'key', 'value'],
)
freeze_qformer_base(model.Qformer)

for param in model.visual_encoder.parameters():
    param.requires_grad = False
for param in model.ln_vision.parameters():
    param.requires_grad = False
for param in model.t5_model.parameters():
    param.requires_grad = False
    param.data = param.data.float()
for param in model.t5_proj.parameters():
    param.requires_grad = True

model = model.to(device)
model.train()

info = get_trainable_params_info(model)
print('Trainable params:', info['trainable_params'])
print('Total params:', info['total_params'])
print('Trainable pct:', round(info['trainable_percentage'], 4))

In [ ]:
# Cell 8: Smoke-train for a short run
optimizer = torch.optim.AdamW(
    [param for param in model.parameters() if param.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
optimizer.zero_grad(set_to_none=True)
loss_history = []
global_step = 0

for epoch in range(EPOCHS):
    print(f'Epoch {epoch + 1}/{EPOCHS}')
    for batch_index, batch in enumerate(train_loader, start=1):
        global_step += 1
        batch['image'] = batch['image'].to(device, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available(), dtype=torch.float16):
            outputs = model(batch)
            loss = outputs['loss'] / GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()

        if batch_index % GRAD_ACCUM_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        loss_value = float(loss.detach().cpu().item() * GRAD_ACCUM_STEPS)
        loss_history.append(loss_value)

        if global_step == 1 or global_step % 10 == 0:
            avg_loss = sum(loss_history[-10:]) / min(len(loss_history), 10)
            print(f'step {global_step:03d} | loss={loss_value:.4f} | avg10={avg_loss:.4f}')

        if global_step >= MAX_STEPS:
            break

    if batch_index % GRAD_ACCUM_STEPS != 0:
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

    if global_step >= MAX_STEPS:
        break

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_path = OUTPUT_DIR / 'blip2_flickr8k_smoke.pt'
torch.save({'model': model.state_dict(), 'step': global_step}, checkpoint_path)
print('Saved checkpoint to', checkpoint_path)

In [ ]:
# Cell 9: Run caption generation on 3 validation images
model.eval()
sample_names = val_names if len(val_names) >= 3 else test_names

with torch.no_grad():
    for image_name in sample_names[:3]:
        image_path = images_dir / image_name
        processed_image = None
        try:
            with Image.open(image_path) as image:
                image = image.convert('RGB')
            processed_image = transform(image).unsqueeze(0).to(device)

            captions = model.generate(
                {'image': processed_image, 'prompt': 'a photo of '},
                use_nucleus_sampling=USE_NUCLEUS_SAMPLING,
                num_beams=NUM_BEAMS,
                max_length=MAX_LENGTH,
                min_length=3,
                top_p=TOP_P,
                temperature=TEMPERATURE,
                repetition_penalty=REPETITION_PENALTY,
                no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
            )

            caption = captions[0].strip() if isinstance(captions, list) else str(captions).strip()
            if not caption:
                print('No caption returned for', image_name)
                caption = '[empty-caption]'

            print(f'Image: {image_name}')
            print(f'Generated:    {caption}')
            print(f'Ground truth: {captions_by_image[image_name][0]}')
            print()
        except Exception as exc:
            print(f'Image: {image_name}')
            print(f'Caption: ERROR - {exc}')
            print('generate() diagnostic checks:')
            print('  tokenizer present:', hasattr(model, 't5_tokenizer'))
            print('  model device:', model.device)
            print('  input shape:', tuple(processed_image.shape) if processed_image is not None else 'N/A')
            print()

model.train()


In [ ]:
# Cell 9b: Captions for another 3 images — change START_IDX to get a different group
# 0 = images 0-2, 3 = images 3-5, 6 = images 6-8, etc.

START_IDX = 3  # <-- change this number and re-run

pool = val_names if len(val_names) > START_IDX else test_names
selected = pool[START_IDX : START_IDX + 3]

print(f'Images {START_IDX} to {START_IDX + len(selected) - 1} of {len(pool)} available\n')

model.eval()
with torch.no_grad():
    for image_name in selected:
        with Image.open(images_dir / image_name) as img:
            img = img.convert('RGB')
        processed = transform(img).unsqueeze(0).to(device)

        captions = model.generate(
            {'image': processed, 'prompt': 'a photo of '},
            use_nucleus_sampling=USE_NUCLEUS_SAMPLING,
            num_beams=NUM_BEAMS,
            max_length=MAX_LENGTH,
            min_length=3,
            top_p=TOP_P,
            temperature=TEMPERATURE,
            repetition_penalty=REPETITION_PENALTY,
            no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
        )

        print(f'Image:        {image_name}')
        print(f'Generated:    {captions[0].strip()}')
        print(f'Ground truth: {captions_by_image[image_name][0]}')
        print()

model.train()


In [ ]:
# Cell 10: Verify architecture is PVT + Q-Former + LoRA (NOT default ViT/EVA)
# Run this AFTER Cell 7 (model must exist). Works after training or before.

import numpy as np
from PIL import Image

def verify_architecture(model):
    """Prove captions come through PVT + Q-Former LoRA, not default BLIP2 ViT."""

    print("=" * 70)
    print("ARCHITECTURE VERIFICATION")
    print("=" * 70)

    # ── 1. Static checks ─────────────────────────────────────────────────
    vit_name = getattr(model, "vit_name", None)
    ve_cls = model.visual_encoder.__class__.__name__
    has_pvt = hasattr(model.visual_encoder, "pvt_encoder")

    print(f"\n[1] Vision encoder")
    print(f"    vit_name attribute:      {vit_name}")
    print(f"    visual_encoder class:    {ve_cls}")
    print(f"    has pvt_encoder attr:    {has_pvt}")
    assert vit_name == "pvt_v2_b2", f"FAIL: vit_name is '{vit_name}', expected 'pvt_v2_b2'"
    assert ("PVT" in ve_cls) or has_pvt, f"FAIL: visual_encoder is not PVT-based: {ve_cls}"
    print("    PASS")

    # ── 2. Q-Former LoRA checks ──────────────────────────────────────────
    lora_all = [n for n, _ in model.Qformer.named_parameters() if "lora" in n.lower()]
    lora_train = [n for n, p in model.Qformer.named_parameters()
                  if "lora" in n.lower() and p.requires_grad]

    print(f"\n[2] Q-Former LoRA")
    print(f"    LoRA param groups:       {len(lora_all)}")
    print(f"    Trainable LoRA groups:   {len(lora_train)}")
    assert len(lora_all) > 0, "FAIL: No LoRA parameters found in Q-Former"
    assert len(lora_train) > 0, "FAIL: LoRA params exist but none are trainable"
    print("    PASS")

    # ── 3. Ensure NO EVA / CLIP / default ViT modules exist ─────────────
    # Use substring match for "eva" and "clip_vit" (short, unlikely to collide).
    # Use exact class-name match for "VisionTransformer" to avoid false positives
    # from PVT classes whose names contain "VisionTransformer" as a substring
    # (e.g. PyramidVisionTransformerV2).
    forbidden_substr = ["eva", "clip_vit"]
    forbidden_exact_cls = {"VisionTransformer"}

    bad_modules = [
        n for n, m in model.named_modules()
        if (any(k.lower() in n.lower() or k.lower() in type(m).__name__.lower()
                for k in forbidden_substr)
            or type(m).__name__ in forbidden_exact_cls)
    ]

    print(f"\n[3] Forbidden module scan (eva / clip_vit / VisionTransformer exact)")
    print(f"    Found:                   {len(bad_modules)}")
    if bad_modules:
        for b in bad_modules[:5]:
            print(f"      - {b}")
    assert len(bad_modules) == 0, f"FAIL: Forbidden ViT/EVA modules present: {bad_modules[:5]}"
    print("    PASS")

    # ── 4. Runtime forward-hook tracing ──────────────────────────────────
    pvt_hits = []
    blocked_hits = []
    hooks = []

    def make_hook(name, mod):
        lname = name.lower()
        cls = mod.__class__.__name__.lower()
        def hook(_m, _inp, _out):
            if "pvt" in lname or "pvt" in cls:
                pvt_hits.append(name)
            if any(k in lname for k in ["eva", "clip_vit"]) or any(k in cls for k in ["eva", "clip_vit"]):
                blocked_hits.append(name)
        return hook

    for n, m in model.named_modules():
        hooks.append(m.register_forward_hook(make_hook(n, m)))

    # Synthetic test image
    dev = next(model.parameters()).device
    arr = np.zeros((224, 224, 3), dtype=np.uint8)
    arr[..., 0] = np.linspace(50, 220, 224, dtype=np.uint8)[None, :]
    arr[..., 1] = np.linspace(220, 40, 224, dtype=np.uint8)[:, None]
    arr[..., 2] = 90
    pil_img = Image.fromarray(arr, "RGB")
    x = transform(pil_img).unsqueeze(0).to(dev)

    was_training = model.training
    model.eval()
    with torch.no_grad():
        caption = model.generate(
            {"image": x, "prompt": "a photo of"},
            use_nucleus_sampling=False,
            num_beams=3,
            max_length=20,
            min_length=1,
        )
    if was_training:
        model.train()

    for h in hooks:
        h.remove()

    print(f"\n[4] Runtime tracing (actual forward pass)")
    print(f"    PVT module activations:  {len(pvt_hits)}")
    print(f"    Blocked activations:     {len(blocked_hits)}")
    assert len(pvt_hits) > 0, "FAIL: No PVT modules fired during forward pass"
    assert len(blocked_hits) == 0, f"FAIL: EVA/CLIP modules fired: {blocked_hits[:5]}"
    print("    PASS")

    # ── 5. Vision output dimension check ─────────────────────────────────
    with torch.no_grad():
        z = model.ln_vision(model.visual_encoder(x))

    print(f"\n[5] Vision output shape")
    print(f"    Shape:                   {tuple(z.shape)}")
    print(f"    Feature dim (last):      {z.shape[-1]}")
    assert z.shape[-1] == 512, f"FAIL: Expected PVT dim=512, got {z.shape[-1]}"
    print("    PASS")

    # ── Summary ──────────────────────────────────────────────────────────
    cap = caption[0] if isinstance(caption, (list, tuple)) else caption
    print("\n" + "=" * 70)
    print("ALL CHECKS PASSED")
    print("=" * 70)
    print(f"  Vision encoder:    PVTv2B2Wrapper (dim=512)")
    print(f"  Q-Former LoRA:     {len(lora_all)} params, {len(lora_train)} trainable")
    print(f"  Default ViT used:  NO")
    print(f"  Sample caption:    {cap}")
    print("=" * 70)

verify_architecture(model)


## What success looks like

You are only looking for a smoke-test pass here:
- the model loads
- loss prints during training
- a checkpoint is saved
- captions print for 3 images

Even if the captions are weak or generic, that is enough to confirm the basic training and inference pipeline is working on Colab free tier.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

# Display validation/test images with captions
# Change START_IDX to view different sets of 3 images (0-2, 3-5, 6-8, etc.)

START_IDX = 6  # Images 6, 7, 8

pool = val_names if len(val_names) > START_IDX else test_names
selected = pool[START_IDX : START_IDX + 3]

num_images = len(selected)
fig, axes = plt.subplots(num_images, 1, figsize=(12, 5 * num_images))

if num_images == 1:
    axes = [axes]

for idx, (ax, image_name) in enumerate(zip(axes, selected)):
    # Load and display image
    img_path = images_dir / image_name
    with Image.open(img_path) as img:
        img = img.convert('RGB')
    
    ax.imshow(img)
    
    # Get ground truth caption
    ground_truth = captions_by_image[image_name][0] if image_name in captions_by_image else "No caption"
    
    # Create title with image info
    caption_text = f"Image: {image_name}\nGround Truth: {ground_truth}"
    ax.set_title(caption_text, fontsize=10, loc='left')
    ax.axis('off')

plt.tight_layout()
plt.show()

print(f'Displayed images {START_IDX} to {START_IDX + len(selected) - 1} of {len(pool)} available')